# K-Means Clustering: Patient Risk Profiling

**Input:** `data/processed/CDC_Diabetes_Dataset_clean.csv` and `data/processed/CDC_Diabetes_Dataset_feature_set_A.csv`

**Purpose:** Apply K-Means clustering to identify patient subgroups with distinct health profiles. Runs across multiple feature subsets (body/demographics, health conditions, lifestyle, socioeconomic, mental/physical health days, full feature set) to expose different structural patterns in the population. Diabetes prevalence is brought in as an interpretation label only — it is not used as a clustering input.

**For each subset:** RobustScaler → PCA(2) → elbow method → silhouette analysis → final K-Means fit → cluster profiling in original feature units.

In [ ]:
import os
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent

DATA_PATH = PROJECT_ROOT / "data" / "processed" / "CDC_Diabetes_Dataset_clean.csv"
DATA_PATH_FEAT_A = PROJECT_ROOT / "data" / "processed" / "CDC_Diabetes_Dataset_feature_set_A.csv"

FIG_DIR = PROJECT_ROOT / "figures" / "results_clustering"
FIG_DIR.mkdir(parents=True, exist_ok=True)


print("Project root directory:", PROJECT_ROOT)
print("Data path exists:", DATA_PATH.exists())
assert DATA_PATH.exists(), f"Data file not found at {DATA_PATH}"
print("Figures directory:", FIG_DIR)

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.preprocessing import StandardScaler, RobustScaler
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
from yellowbrick.cluster import KElbowVisualizer
from sklearn.utils import resample


In [ ]:
pd.set_option("display.max_columns", None)
pd.set_option("display.width", None)
pd.set_option("display.max_colwidth", None)

In [ ]:
# ---------------------------------------
# Setup: imports, paths, utilities
# ---------------------------------------

def savefig(name: str):
    """
    Save the current matplotlib figure to the figures directory
    at publication-quality resolution.
    """
    plt.savefig(FIG_DIR / name, dpi=300, bbox_inches="tight")
    print(f"Figure saved: {FIG_DIR / name}")

In [ ]:
# load data from csv
df_cdc_clean = pd.read_csv(DATA_PATH)
print("Data loaded successfully.")
print("Top 5 rows:")
#df_cdc_clean.head()

df_cdc_featA = pd.read_csv(DATA_PATH_FEAT_A)
print("Data loaded successfully.")
print("Top 5 rows:")
df_cdc_featA.head()




In [ ]:
df_cdc_clean.describe()

In [ ]:
# Create base dataset from the dataset used in classification
df_featA = df_cdc_featA.copy()

engineered_cols = [
    'RiskFactorCount', 
    'BMI_PhysActivity',
    "Age_HighBP",
    "Log1p_MentHlth",
    "Log1p_PhysHlth"
]

# Create base dataset (without engineered features
df_base = df_featA.drop(columns = [c for c in engineered_cols if c in df_featA.columns]).copy()

print("df_featA shape:", df_featA.shape)
print("df_base  shape:", df_base.shape)

# sanity check: split column should exist + proportions
print("\nSplit counts (featA):")
print(df_featA["split"].value_counts())

print("\nSplit counts (base):")
print(df_base["split"].value_counts())

# confirm engineered columns presence/absence
print("\nEngineered columns present in featA:", [c for c in engineered_cols if c in df_featA.columns])
print("Engineered columns present in base :", [c for c in engineered_cols if c in df_base.columns])

Feature set definitions are provided in the next code cell.

In [ ]:
# ============================================================
# Feature set definitions (used across all clustering runs)
# ============================================================

# Target variable:
# Defined here for later INTERPRETATION only.
# not used as input to any clustering model.
TARGET = "Diabetes_binary"

feature_sets = {
    # A) "Health metrics / conditions"
    "A_health_conditions": [
        "HighBP", "HighChol", "CholCheck", "Stroke", "HeartDiseaseorAttack",
        "AnyHealthcare", "NoDocbcCost", "DiffWalk", "GenHlth"
    ],

    # B) "Lifestyle / behaviours"
    "B_lifestyle": [
        "Smoker", "PhysActivity", "Fruits", "Veggies", "HvyAlcoholConsump"
    ],

    # C) "Socioeconomic"
    "C_socioeconomic": [
        "Education", "Income"
    ],

    # D) "Body composition / demographics core"
    "D_body_demo": [
        "BMI", "Age", "Sex"
    ],

    # E) "Mental + physical health days"
    "E_mental_physical_days": [
        "MentHlth", "PhysHlth"
    ],
}

# ------------------------------------------------------------
# Sanity check: ensure all features exist in the base dataset
# ------------------------------------------------------------
DATA_FOR_CHECK = df_base

missing = {
    k: [c for c in cols if c not in DATA_FOR_CHECK.columns]
    for k, cols in feature_sets.items()
}
missing = {k: v for k, v in missing.items() if len(v) > 0}

print("Target present in dataset:", TARGET in DATA_FOR_CHECK.columns)
print("Defined feature sets:", list(feature_sets.keys()))
print("Any missing columns:", bool(missing))

if missing:
    for k, v in missing.items():
        print(f"{k} missing columns: {v}")

# Optional: show feature set sizes
for k, cols in feature_sets.items():
    print(f"{k}: {len(cols)} features")




**Clustering Pipeline D**

In [ ]:
# ==============================
# Define clustering view for feature set D
# ==============================

# Choose dataset (primary clustering on base features)
DF_CLUSTER = df_base

# Use TRAIN only (same split as classification)
df_train = DF_CLUSTER[DF_CLUSTER["split"] == "train"].copy()

# Choose feature set (from feature_sets dict defined above)
FEATURE_SET_NAME = "D_body_demo"
FEATURE_COLS = feature_sets[FEATURE_SET_NAME]

# Build raw clustering matrix (NO transforms yet)
X_train_raw = df_train[FEATURE_COLS].copy()

# Sanity checks
print("Dataset used        : df_base")
print("Rows used           : train only")
print("Feature set         :", FEATURE_SET_NAME)
print("Feature columns     :", FEATURE_COLS)
print("X_train_raw shape   :", X_train_raw.shape)
print("Missing values      :", int(X_train_raw.isna().sum().sum()))
print("Unique values       :", {c: int(X_train_raw[c].nunique()) for c in FEATURE_COLS})

In [ ]:
# confirm outliers in the dataset through boxplot, already observed in EDA
cols = FEATURE_COLS

plt.figure(figsize=(8,4))
plt.boxplot([df_train[c].values for c in cols], tick_labels=cols, showfliers=True)
plt.title("Boxplots (TRAIN, raw values) — outliers visible")
plt.tight_layout()
savefig("boxplot_outlier_check_D.png")
plt.show()


In [ ]:
df_train[["BMI"]].describe(percentiles=[0.01,0.05,0.5,0.95,0.99]).T

In [ ]:
# Apply scaler to train data - fit on train only
scaler = RobustScaler()
X_train_scaled = scaler.fit_transform(X_train_raw)

print("X_train_scaled shape:", X_train_scaled.shape)
print("Scaled median (approx 0):",  __import__("numpy").median(X_train_scaled, axis=0))
print("IQR (approx 1):", __import__("numpy").percentile(X_train_scaled, 75, axis=0) - __import__("numpy").percentile(X_train_scaled, 25, axis=0))

In [ ]:
# PCA

pca = PCA(random_state=42)
X_train_pca = pca.fit_transform(X_train_scaled)

expl = pca.explained_variance_ratio_
cum  = np.cumsum(expl)

print("Explained variance ratio:", np.round(expl, 4))
print("Cumulative explained variance:", np.round(cum, 4))
print("X_train_pca shape:", X_train_pca.shape)

In [ ]:
# plot PCA explained variance

explained = pca.explained_variance_ratio_
components = np.arange(1, len(explained) + 1)

plt.figure()
plt.plot(components, explained, marker="o", label="Individual variance")
plt.plot(components, np.cumsum(explained), marker="o", linestyle="--", label="Cumulative variance")
plt.xlabel("Principal Component")
plt.ylabel("Explained Variance Ratio")
plt.title("PCA Explained Variance (TRAIN, Robust-scaled)")
plt.xticks(components)
plt.legend()
plt.tight_layout()
savefig("pca_explained_variance_D.png")
plt.show()

In [ ]:
pca_2 = PCA(n_components=2, random_state=42)
X_train_pca_2 = pca_2.fit_transform(X_train_scaled)

print("X_train_pca_2 shape:", X_train_pca_2.shape)
print("Explained variance (2 PCs):", pca_2.explained_variance_ratio_)
print("Cumulative variance:", pca_2.explained_variance_ratio_.sum())

In [ ]:
# Plot of PC1 vs PC2
plt.figure(figsize=(6,5))
plt.scatter(
    X_train_pca_2[:, 0],
    X_train_pca_2[:, 1],
    s=5,
    alpha=0.3
)
plt.xlabel("PC1")
plt.ylabel("PC2")
plt.title("PCA projection (2 components) — TRAIN data")
plt.tight_layout()
savefig("pca_projection_D.png")
plt.show()

In [ ]:
# Elbow method

ks = list(range(2, 11))
inertias = []

for k in ks:
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    km.fit(X_train_pca_2)
    inertias.append(km.inertia_)
    print(f"k={k:2d} | inertia={km.inertia_:.2f}")

elbow_df = pd.DataFrame({"k": ks, "inertia": inertias})

plt.figure()
plt.plot(elbow_df["k"], elbow_df["inertia"], marker="o")
plt.xlabel("Number of clusters (k)")
plt.ylabel("Inertia (WCSS)")
plt.title("Elbow Method — KMeans on PCA(2) space (df_base, Feature set D, TRAIN)")
plt.tight_layout()
plt.savefig(FIG_DIR / "kmeans_elbow_df_base_D.png", dpi=300)
plt.show()

elbow_df

In [ ]:
#Silhouette score

ks = list(range(2, 11))
sil_scores = []

# sample for speed
rng = np.random.RandomState(42)
sample_n = 20000 if X_train_pca_2.shape[0] > 20000 else X_train_pca_2.shape[0]
idx = rng.choice(X_train_pca_2.shape[0], size=sample_n, replace=False)
X_samp = X_train_pca_2[idx]

for k in ks:
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    km.fit(X_train_pca_2)

    sil = silhouette_score(X_samp, km.predict(X_samp))
    sil_scores.append(sil)
    print(f"k={k:2d} | silhouette(sample)={sil:.4f}")

sil_df = pd.DataFrame({"k": ks, "silhouette_sample": sil_scores})

plt.figure()
plt.plot(sil_df["k"], sil_df["silhouette_sample"], marker="o")
plt.xlabel("Number of clusters (k)")
plt.ylabel("Silhouette score (sample)")
plt.title("Silhouette (sample) — KMeans on PCA(2) space (df_base, Feature set D, TRAIN)")
plt.tight_layout()
plt.savefig(FIG_DIR / "kmeans_silhouette_df_base_D.png", dpi=300)
plt.show()

sil_df

In [ ]:
K_FINAL = 3

km_final = KMeans(n_clusters=K_FINAL, random_state=42, n_init=10)
train_clusters = km_final.fit_predict(X_train_pca_2)

print("K_FINAL:", K_FINAL)
print("Train cluster counts:", dict(zip(*np.unique(train_clusters, return_counts=True))))

centroids = km_final.cluster_centers_
cmap = plt.get_cmap("tab10")

plt.figure(figsize=(6,5))

for k in range(K_FINAL):
    mask = train_clusters == k
    plt.scatter(
        X_train_pca_2[mask, 0],
        X_train_pca_2[mask, 1],
        s=10,
        alpha=0.6,
        color=cmap(k),
        label=f"Cluster {k}"
    )

# Centroids (X markers)
plt.scatter(
    centroids[:, 0],
    centroids[:, 1],
    marker="X",
    s=200,
    c=[cmap(i) for i in range(K_FINAL)],
    edgecolor="black",
    linewidths=1.5
)


plt.xlabel("PC1")
plt.ylabel("PC2")
plt.title("KMeans Clusters (k=3) — PCA(2) space (df_base, Feature set D, TRAIN)")
plt.legend(markerscale=2, frameon=True)
plt.tight_layout()
savefig("kmeans_clusters_df_base_D_k3_centroids.png")
plt.show()

In [ ]:
# Attach clusters to df_train
# profile clusteres in the original feature units
df_train_clusters = df_train.copy()
df_train_clusters["cluster"] = train_clusters

# Quick sanity check - how many records into each cluster?
print(df_train_clusters["cluster"].value_counts())

# C0 approx 50%, C2 30%, C1 20% (approx)
# no cluster is too small
# All clustes have good representation


In [ ]:
# profile each cluster using original variables
profile_cols = ["BMI", "Age", "Sex"]

cluster_profile = (
    df_train_clusters
    .groupby("cluster")[profile_cols]
    .agg(["mean", "median"])
    .round(2)
)

cluster_profile

# table results shows mean and median for each cluster
# understand characteristics for each cluster

In [ ]:
# Bring Diabetes_binary back in for interpretation only
df_train_clusters["Diabetes_binary"] = df_featA.loc[df_train_clusters.index, "Diabetes_binary"]

diabetes_by_cluster = (
    df_train_clusters
    .groupby("cluster")["Diabetes_binary"]
    .mean()
    .round(3)
)
diabetes_by_cluster

# shows diabetes prevalence for each cluster
# used to understand which cluster is higher risk

**Clustering Pipeline A**
- Most cells from clustering pipeline D will remain the same, except changing D-->A

In [ ]:
# ==============================
# Define clustering view - feature set A
# ==============================

DF_CLUSTER = df_base
df_train_A = DF_CLUSTER[DF_CLUSTER["split"] == "train"].copy()

FEATURE_SET_NAME = "A_health_conditions"
FEATURE_COLS_A = feature_sets[FEATURE_SET_NAME]

X_train_raw_A = df_train[FEATURE_COLS_A].copy()

print("Dataset used        : df_base")
print("Rows used           : train only")
print("Feature set         :", FEATURE_SET_NAME)
print("Feature columns     :", FEATURE_COLS_A)
print("X_train_raw shape   :", X_train_raw_A.shape)
print("Missing values      :", int(X_train_raw_A.isna().sum().sum()))

In [ ]:
# confirm outliers in the dataset through boxplot, already observed in EDA
cols = FEATURE_COLS_A

plt.figure(figsize=(8,4))
plt.boxplot([df_train_A[c].values for c in cols], tick_labels=cols, showfliers=True)
plt.title("Boxplots (TRAIN, raw values) — outliers visible")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
savefig("boxplot_outlier_check_A.png")
plt.show()


In [ ]:
summary_A = df_train_A[FEATURE_COLS_A].describe(
    percentiles=[0.01, 0.05, 0.5, 0.95, 0.99]
).T
summary_A

In [ ]:
# Apply scaler to train data - fit on train only

scaler_A = RobustScaler()
X_train_scaled_A = scaler_A.fit_transform(X_train_raw_A)

print("X_train_scaled_A shape:", X_train_scaled_A.shape)
print("Scaled medians:", np.median(X_train_scaled_A, axis=0))
print("Scaled IQRs:", np.percentile(X_train_scaled_A, 75, axis=0) - np.percentile(X_train_scaled_A, 25, axis=0))

In [ ]:
# PCA

pca_A = PCA(random_state=42)
X_train_pca_A = pca_A.fit_transform(X_train_scaled_A)

expl = pca_A.explained_variance_ratio_
cum = np.cumsum(expl)

print("Explained variance ratio (first 9):", np.round(expl, 4))
print("Cumulative variance:", np.round(cum, 4))

# Scree / cumulative plot
components = np.arange(1, len(expl) + 1)
plt.figure()
plt.plot(components, expl, marker="o", label="Individual variance")
plt.plot(components, cum, marker="o", linestyle="--", label="Cumulative variance")
plt.xlabel("Principal Component")
plt.ylabel("Explained Variance Ratio")
plt.title("PCA Explained Variance — Feature set A (TRAIN, Robust-scaled)")
plt.xticks(components)
plt.legend()
plt.tight_layout()
savefig("pca_explained_variance_df_base_A.png")
plt.show()

In [ ]:
# PCA projection (PC1 vs PC2)

pca_A2 = PCA(n_components=2, random_state=42)
X_train_pca_A2 = pca_A2.fit_transform(X_train_scaled_A)

print("X_train_pca_A2 shape:", X_train_pca_A2.shape)
print("Explained variance (2 PCs):", pca_A2.explained_variance_ratio_)
print("Cumulative variance:", pca_A2.explained_variance_ratio_.sum())

plt.figure(figsize=(6,5))
plt.scatter(X_train_pca_A2[:,0], X_train_pca_A2[:,1], s=5, alpha=0.3)
plt.xlabel("PC1")
plt.ylabel("PC2")
plt.title("PCA projection (2 components) — Feature set A (TRAIN)")
plt.tight_layout()
savefig("pca_scatter_df_base_A_train.png")
plt.show()

In [ ]:
# Elbow method

ks = list(range(2, 11))
inertias = []

for k in ks:
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    km.fit(X_train_pca_A2)
    inertias.append(km.inertia_)
    print(f"k={k:2d} | inertia={km.inertia_:.2f}")

elbow_A_df = pd.DataFrame({"k": ks, "inertia": inertias})

plt.figure()
plt.plot(elbow_A_df["k"], elbow_A_df["inertia"], marker="o")
plt.xlabel("Number of clusters (k)")
plt.ylabel("Inertia (WCSS)")
plt.title("Elbow — KMeans on PCA(2) (df_base, Feature set A, TRAIN)")
plt.tight_layout()
savefig( "kmeans_elbow_df_base_A.png")
plt.show()

In [ ]:
# Silhouette 


ks = list(range(2, 11))
sil_scores = []

# sample for speed
rng = np.random.RandomState(42)
sample_n = 20000 if X_train_pca_A2.shape[0] > 20000 else X_train_pca_A2.shape[0]
idx = rng.choice(X_train_pca_A2.shape[0], size=sample_n, replace=False)
X_samp = X_train_pca_A2[idx]

for k in ks:
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    km.fit(X_train_pca_A2)
    sil = silhouette_score(X_samp, km.predict(X_samp))
    sil_scores.append(sil)
    print(f"k={k:2d} | silhouette(sample)={sil:.4f}")

sil_A_df = pd.DataFrame({"k": ks, "silhouette_sample": sil_scores})

plt.figure()
plt.plot(sil_A_df["k"], sil_A_df["silhouette_sample"], marker="o")
plt.xlabel("Number of clusters (k)")
plt.ylabel("Silhouette score (sample)")
plt.title("Silhouette (sample) — KMeans on PCA(2) (df_base, Feature set A, TRAIN)")
plt.tight_layout()
savefig("kmeans_silhouette_df_base_A.png")
plt.show()

sil_A_df

In [ ]:

cmap = plt.get_cmap("tab10")
Ks_to_compare = [3, 4, 5]

for K in Ks_to_compare:
    km = KMeans(n_clusters=K, random_state=42, n_init=10)
    labels = km.fit_predict(X_train_pca_A2)
    centroids = km.cluster_centers_

    print(f"\nK={K} cluster counts:", dict(zip(*np.unique(labels, return_counts=True))))

    plt.figure(figsize=(6,5))
    for k in range(K):
        mask = labels == k
        plt.scatter(
            X_train_pca_A2[mask, 0],
            X_train_pca_A2[mask, 1],
            s=10, alpha=0.6,
            color=cmap(k),
            label=f"Cluster {k}"
        )

    # centroids
    plt.scatter(
        centroids[:, 0], centroids[:, 1],
        marker="X", s=220,
        c=[cmap(i) for i in range(K)],
        edgecolor="black", linewidths=1.5
    )
    

    plt.xlabel("PC1")
    plt.ylabel("PC2")
    plt.title(f"KMeans (k={K}) — Feature set A (TRAIN, PCA2)")
    plt.legend(markerscale=2, frameon=True, ncol=2)
    plt.tight_layout()
    savefig(f"kmeans_clusters_df_base_A_k{K}_centroids.png")
    plt.show()

In [ ]:
# Comparison table for K = 3, 4, 5 using existing elbow/silhouette results
Ks_to_compare = [3, 4, 5]
rows = []

for K in Ks_to_compare:
    km = KMeans(n_clusters=K, random_state=42, n_init=10)
    labels = km.fit_predict(X_train_pca_A2)
    counts = dict(zip(*np.unique(labels, return_counts=True)))

    inertia = float(elbow_A_df.loc[elbow_A_df["k"] == K, "inertia"].iloc[0])
    sil = float(sil_A_df.loc[sil_A_df["k"] == K, "silhouette_sample"].iloc[0])

    rows.append({"K": K, "inertia": inertia, "silhouette_sample": sil, "counts": counts})

compare_A = pd.DataFrame(rows).sort_values("K")
compare_A

In [ ]:
# Define final clusters for feature set A (K=4)
K_A = 4
km_A = KMeans(n_clusters=K_A, random_state=42, n_init=10)
clusters_A = km_A.fit_predict(X_train_pca_A2)

In [ ]:
# Attach clusters to df_train
# profile clusteres in the original feature units
df_train_A_clusters = df_train_A.copy()
df_train_A_clusters["cluster_A"] = clusters_A

print(df_train_A_clusters["cluster_A"].value_counts())

# Profile health-condition features (means = prevalence for 0/1 vars)
profile_A = (
    df_train_A_clusters
    .groupby("cluster_A")[FEATURE_COLS_A]
    .mean()
    .round(3)
)

profile_A

In [ ]:
# Add target back in for interpretation only (from df_featA)
df_train_A_clusters[TARGET] = df_featA.loc[df_train_A_clusters.index, TARGET]

diabetes_by_cluster_A = (
    df_train_A_clusters
    .groupby("cluster_A")[TARGET]
    .mean()
    .round(3)
)

diabetes_by_cluster_A

**Clustering pipeline B, C, E**

In [ ]:
# --- Settings ---
APP_SETS = ["B_lifestyle", "C_socioeconomic", "E_mental_physical_days"]
KS = list(range(2, 11))
SIL_SAMPLE_N = 20000
RANDOM_STATE = 42

def run_feature_set_appendix(fs_name, fit_final=True, max_k_auto=6):
    """
    Runs: train-only -> RobustScaler -> PCA(2) -> elbow + silhouette(sample) -> auto-pick k
    Optionally fits final KMeans and outputs profiling tables + diabetes prevalence.
    """
    # 1) View
    DF_CLUSTER = df_base
    df_train_fs = DF_CLUSTER[DF_CLUSTER["split"] == "train"].copy()

    cols = feature_sets[fs_name]
    X_raw = df_train_fs[cols].copy()

    # Basic checks
    miss = int(X_raw.isna().sum().sum())
    if miss != 0:
        raise ValueError(f"{fs_name}: missing values found ({miss}). Handle before clustering.")

    # 2) Scale
    scaler = RobustScaler()
    X_scaled = scaler.fit_transform(X_raw)

    # 3) PCA(2) (or 2D if only 2 features)
    n_pca = min(2, X_scaled.shape[1])
    pca = PCA(n_components=n_pca, random_state=RANDOM_STATE)
    X_pca = pca.fit_transform(X_scaled)

    # Save PCA scatter (optional but useful)
    plt.figure(figsize=(6,5))
    plt.scatter(X_pca[:,0], X_pca[:,1], s=5, alpha=0.3)
    plt.xlabel("PC1")
    plt.ylabel("PC2")
    plt.title(f"PCA projection (2 components) — {fs_name} (TRAIN)")
    plt.tight_layout()
    plt.savefig(FIG_DIR / f"pca_scatter_df_base_{fs_name}_train.png", dpi=300)
    plt.show()

    # 4) Elbow (inertia)
    inertias = []
    for k in KS:
        km = KMeans(n_clusters=k, random_state=RANDOM_STATE, n_init=10)
        km.fit(X_pca)
        inertias.append(km.inertia_)

    elbow_df = pd.DataFrame({"k": KS, "inertia": inertias})
    plt.figure()
    plt.plot(elbow_df["k"], elbow_df["inertia"], marker="o")
    plt.xlabel("Number of clusters (k)")
    plt.ylabel("Inertia (WCSS)")
    plt.title(f"Elbow — KMeans on PCA(2) ({fs_name}, TRAIN)")
    plt.tight_layout()
    plt.savefig(FIG_DIR / f"kmeans_elbow_df_base_{fs_name}.png", dpi=300)
    plt.show()

    # 5) Silhouette (sample)
    rng = np.random.RandomState(RANDOM_STATE)
    n = X_pca.shape[0]
    sample_n = SIL_SAMPLE_N if n > SIL_SAMPLE_N else n
    idx = rng.choice(n, size=sample_n, replace=False)
    X_samp = X_pca[idx]

    sils = []
    for k in KS:
        km = KMeans(n_clusters=k, random_state=RANDOM_STATE, n_init=10)
        km.fit(X_pca)
        sil = silhouette_score(X_samp, km.predict(X_samp))
        sils.append(sil)

    sil_df = pd.DataFrame({"k": KS, "silhouette_sample": sils})
    plt.figure()
    plt.plot(sil_df["k"], sil_df["silhouette_sample"], marker="o")
    plt.xlabel("Number of clusters (k)")
    plt.ylabel("Silhouette score (sample)")
    plt.title(f"Silhouette (sample) — KMeans on PCA(2) ({fs_name}, TRAIN)")
    plt.tight_layout()
    plt.savefig(FIG_DIR / f"kmeans_silhouette_df_base_{fs_name}.png", dpi=300)
    plt.show()

    # 6) Choose K (appendix heuristic)
    # We cap k to avoid over-segmentation in appendix.
    sil_df_cap = sil_df[sil_df["k"] <= max_k_auto]
    k_auto = int(sil_df_cap.loc[sil_df_cap["silhouette_sample"].idxmax(), "k"])

    print(f"\n[{fs_name}] PCA(2) cumulative variance: {pca.explained_variance_ratio_.sum():.3f}")
    print(f"[{fs_name}] Auto K (max {max_k_auto}): {k_auto}")
    print(f"[{fs_name}] Best silhouette within cap: {sil_df_cap['silhouette_sample'].max():.3f}")

    outputs = {
        "fs_name": fs_name,
        "cols": cols,
        "pca_cumvar": float(pca.explained_variance_ratio_.sum()),
        "elbow_df": elbow_df,
        "sil_df": sil_df,
        "k_auto": k_auto,
    }

    if not fit_final:
        return outputs

    # 7) Fit final KMeans + plot + centroids
    km_final = KMeans(n_clusters=k_auto, random_state=RANDOM_STATE, n_init=10)
    labels = km_final.fit_predict(X_pca)
    centroids = km_final.cluster_centers_
    cmap = plt.get_cmap("tab10")

    plt.figure(figsize=(6,5))
    for k in range(k_auto):
        m = labels == k
        plt.scatter(X_pca[m,0], X_pca[m,1], s=10, alpha=0.6, color=cmap(k), label=f"Cluster {k}")
    plt.scatter(centroids[:,0], centroids[:,1], marker="X", s=220,
                c=[cmap(i) for i in range(k_auto)], edgecolor="black", linewidths=1.5)
    plt.xlabel("PC1"); plt.ylabel("PC2")
    plt.title(f"KMeans (k={k_auto}) — {fs_name} (TRAIN, PCA2)")
    plt.legend(markerscale=2, frameon=True, ncol=2)
    plt.tight_layout()
    plt.savefig(FIG_DIR / f"kmeans_clusters_df_base_{fs_name}_k{k_auto}_centroids.png", dpi=300)
    plt.show()

    # 8) Profile in original feature space (means; for binary -> prevalence)
    df_train_out = df_train_fs.copy()
    df_train_out[f"cluster_{fs_name}"] = labels
    profile = df_train_out.groupby(f"cluster_{fs_name}")[cols].mean().round(3)

    # 9) Diabetes prevalence (interpretation only)
    if TARGET in df_featA.columns:
        df_train_out[TARGET] = df_featA.loc[df_train_out.index, TARGET]
        diab = df_train_out.groupby(f"cluster_{fs_name}")[TARGET].mean().round(3)
    else:
        diab = None

    outputs.update({
        "k_final": k_auto,
        "profile": profile,
        "diabetes_prevalence": diab,
        "cluster_counts": df_train_out[f"cluster_{fs_name}"].value_counts().to_dict()
    })
    return outputs


# --- Run appendix sets ---
appendix_results = {}
for fs in APP_SETS:
    appendix_results[fs] = run_feature_set_appendix(fs_name=fs, fit_final=True, max_k_auto=6)

# Quick peek: print diabetes prevalence per set
for fs, res in appendix_results.items():
    print(f"\n=== {fs} ===")
    print("k_final:", res["k_final"])
    print("cluster_counts:", res["cluster_counts"])
    print("diabetes_prevalence:\n", res["diabetes_prevalence"])

**Clustering pipeline on full base and engineered dataset**

In [ ]:

# Full feature lists 
DROP_COLS = {"split", TARGET}

# Base full features
FULL_COLS_BASE = [c for c in df_base.columns if c not in DROP_COLS]

# Engineered full features
FULL_COLS_FEATA = [c for c in df_featA.columns if c not in DROP_COLS]

print("FULL_COLS_BASE:", len(FULL_COLS_BASE))
print("FULL_COLS_FEATA:", len(FULL_COLS_FEATA))

# sanity: engineered should be base + 5 (usually)
extra_in_featA = sorted(set(FULL_COLS_FEATA) - set(FULL_COLS_BASE))
print("Extra columns in featA (vs base):", extra_in_featA)

In [ ]:

# --- TRAIN only ---
df_train_full_base = df_base[df_base["split"] == "train"].copy()
X_raw_full_base = df_train_full_base[FULL_COLS_BASE].copy()

print("X_raw_full_base shape:", X_raw_full_base.shape)
print("Missing values:", int(X_raw_full_base.isna().sum().sum()))

# --- Scale ---
scaler_full_base = RobustScaler()
X_scaled_full_base = scaler_full_base.fit_transform(X_raw_full_base)

# --- PCA (inspect) ---
pca_full_base = PCA(random_state=42)
X_pca_full_base = pca_full_base.fit_transform(X_scaled_full_base)

expl = pca_full_base.explained_variance_ratio_
cum = np.cumsum(expl)

print("Cum var @2 PCs:", round(cum[1], 4))
print("Cum var @5 PCs:", round(cum[4], 4))
print("Cum var @10 PCs:", round(cum[9], 4))

# Scree plot
components = np.arange(1, len(expl) + 1)
plt.figure()
plt.plot(components, expl, marker="o", label="Individual variance")
plt.plot(components, cum, marker="o", linestyle="--", label="Cumulative variance")
plt.xlabel("Principal Component")
plt.ylabel("Explained Variance Ratio")
plt.title("PCA Explained Variance — FULL base feature set (TRAIN, Robust-scaled)")
plt.xticks(components)
plt.legend()
plt.tight_layout()
plt.savefig(FIG_DIR / "pca_explained_variance_full_base.png", dpi=300)
plt.show()

# --- PCA(2) for clustering/visualisation ---
pca2_full_base = PCA(n_components=2, random_state=42)
X_pca2_full_base = pca2_full_base.fit_transform(X_scaled_full_base)

plt.figure(figsize=(6,5))
plt.scatter(X_pca2_full_base[:,0], X_pca2_full_base[:,1], s=5, alpha=0.3)
plt.xlabel("PC1")
plt.ylabel("PC2")
plt.title("PCA projection (2 components) — FULL base feature set (TRAIN)")
plt.tight_layout()
plt.savefig(FIG_DIR / "pca_scatter_full_base_train.png", dpi=300)
plt.show()

print("Explained variance (2 PCs):", pca2_full_base.explained_variance_ratio_)
print("Cumulative variance (2 PCs):", pca2_full_base.explained_variance_ratio_.sum())

In [ ]:
ks = list(range(2, 11))
inertias = []

for k in ks:
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    km.fit(X_pca2_full_base)
    inertias.append(km.inertia_)
    print(f"k={k:2d} | inertia={km.inertia_:.2f}")

elbow_full_base_df = pd.DataFrame({"k": ks, "inertia": inertias})

plt.figure()
plt.plot(elbow_full_base_df["k"], elbow_full_base_df["inertia"], marker="o")
plt.xlabel("Number of clusters (k)")
plt.ylabel("Inertia (WCSS)")
plt.title("Elbow — KMeans on PCA(2) (FULL base, TRAIN)")
plt.tight_layout()
plt.savefig(FIG_DIR / "kmeans_elbow_full_base.png", dpi=300)
plt.show()

In [ ]:


ks = list(range(2, 11))
sil_scores = []

rng = np.random.RandomState(42)
sample_n = 20000 if X_pca2_full_base.shape[0] > 20000 else X_pca2_full_base.shape[0]
idx = rng.choice(X_pca2_full_base.shape[0], size=sample_n, replace=False)
X_samp = X_pca2_full_base[idx]

for k in ks:
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    km.fit(X_pca2_full_base)
    sil = silhouette_score(X_samp, km.predict(X_samp))
    sil_scores.append(sil)
    print(f"k={k:2d} | silhouette(sample)={sil:.4f}")

sil_full_base_df = pd.DataFrame({"k": ks, "silhouette_sample": sil_scores})

plt.figure()
plt.plot(sil_full_base_df["k"], sil_full_base_df["silhouette_sample"], marker="o")
plt.xlabel("Number of clusters (k)")
plt.ylabel("Silhouette score (sample)")
plt.title("Silhouette (sample) — KMeans on PCA(2) (FULL base, TRAIN)")
plt.tight_layout()
plt.savefig(FIG_DIR / "kmeans_silhouette_full_base.png", dpi=300)
plt.show()

In [ ]:
K_FULL_BASE = 4

km_full_base = KMeans(n_clusters=K_FULL_BASE, random_state=42, n_init=10)
clusters_full_base = km_full_base.fit_predict(X_pca2_full_base)

print("K_FULL_BASE:", K_FULL_BASE)
print("Cluster counts:", dict(zip(*np.unique(clusters_full_base, return_counts=True))))

centroids = km_full_base.cluster_centers_
cmap = plt.get_cmap("tab10")

plt.figure(figsize=(6,5))
for k in range(K_FULL_BASE):
    m = clusters_full_base == k
    plt.scatter(
        X_pca2_full_base[m, 0],
        X_pca2_full_base[m, 1],
        s=10, alpha=0.6,
        color=cmap(k),
        label=f"Cluster {k}"
    )

plt.scatter(
    centroids[:, 0], centroids[:, 1],
    marker="X", s=220,
    c=[cmap(i) for i in range(K_FULL_BASE)],
    edgecolor="black", linewidths=1.5
)


plt.xlabel("PC1")
plt.ylabel("PC2")
plt.title("KMeans Clusters (k=4) — FULL base feature set (TRAIN)")
plt.legend(markerscale=2, frameon=True, ncol=2)
plt.tight_layout()
plt.savefig(FIG_DIR / "kmeans_clusters_full_base_k4_centroids.png", dpi=300)
plt.show()

In [ ]:
# Attach clusters back to train data
df_train_full_base_out = df_train_full_base.copy()
df_train_full_base_out["cluster_full_base"] = clusters_full_base

# Profile ALL base features (means; for binary -> prevalence)
profile_full_base = (
    df_train_full_base_out
    .groupby("cluster_full_base")[FULL_COLS_BASE]
    .mean()
    .round(3)
)

profile_full_base

In [ ]:
# Diabetes prevalence (interpretation only)
df_train_full_base_out[TARGET] = df_featA.loc[df_train_full_base_out.index, TARGET]

diabetes_full_base = (
    df_train_full_base_out
    .groupby("cluster_full_base")[TARGET]
    .mean()
    .round(3)
)

diabetes_full_base

**Clustering on engineered dataset for comparison**

In [ ]:
# --- TRAIN only ---
df_train_full_featA = df_featA[df_featA["split"] == "train"].copy()
X_raw_full_featA = df_train_full_featA[FULL_COLS_FEATA].copy()

print("X_raw_full_featA shape:", X_raw_full_featA.shape)
print("Missing values:", int(X_raw_full_featA.isna().sum().sum()))

# --- Scale ---
scaler_full_featA = RobustScaler()
X_scaled_full_featA = scaler_full_featA.fit_transform(X_raw_full_featA)

# --- PCA (inspect) ---
pca_full_featA = PCA(random_state=42)
X_pca_full_featA = pca_full_featA.fit_transform(X_scaled_full_featA)

expl = pca_full_featA.explained_variance_ratio_
cum = np.cumsum(expl)

print("Cum var @2 PCs:", round(cum[1], 4))
print("Cum var @5 PCs:", round(cum[4], 4))
print("Cum var @10 PCs:", round(cum[9], 4))

# Scree plot
components = np.arange(1, len(expl) + 1)
plt.figure()
plt.plot(components, expl, marker="o", label="Individual variance")
plt.plot(components, cum, marker="o", linestyle="--", label="Cumulative variance")
plt.xlabel("Principal Component")
plt.ylabel("Explained Variance Ratio")
plt.title("PCA Explained Variance — FULL engineered feature set (TRAIN, Robust-scaled)")
plt.xticks(components)
plt.legend()
plt.tight_layout()
plt.savefig(FIG_DIR / "pca_explained_variance_full_featA.png", dpi=300)
plt.show()

# --- PCA(2) for clustering/visualisation ---
pca2_full_featA = PCA(n_components=2, random_state=42)
X_pca2_full_featA = pca2_full_featA.fit_transform(X_scaled_full_featA)

plt.figure(figsize=(6,5))
plt.scatter(X_pca2_full_featA[:,0], X_pca2_full_featA[:,1], s=5, alpha=0.3)
plt.xlabel("PC1")
plt.ylabel("PC2")
plt.title("PCA projection (2 components) — FULL engineered feature set (TRAIN)")
plt.tight_layout()
plt.savefig(FIG_DIR / "pca_scatter_full_featA_train.png", dpi=300)
plt.show()

print("Explained variance (2 PCs):", pca2_full_featA.explained_variance_ratio_)
print("Cumulative variance (2 PCs):", pca2_full_featA.explained_variance_ratio_.sum())

In [ ]:
ks = list(range(2, 11))
inertias = []

for k in ks:
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    km.fit(X_pca2_full_featA)
    inertias.append(km.inertia_)
    print(f"k={k:2d} | inertia={km.inertia_:.2f}")

elbow_full_featA_df = pd.DataFrame({"k": ks, "inertia": inertias})

plt.figure()
plt.plot(elbow_full_featA_df["k"], elbow_full_featA_df["inertia"], marker="o")
plt.xlabel("Number of clusters (k)")
plt.ylabel("Inertia (WCSS)")
plt.title("Elbow — KMeans on PCA(2) (FULL engineered, TRAIN)")
plt.tight_layout()
plt.savefig(FIG_DIR / "kmeans_elbow_full_featA.png", dpi=300)
plt.show()

In [ ]:
ks = list(range(2, 11))
sil_scores = []

rng = np.random.RandomState(42)
sample_n = 20000 if X_pca2_full_featA.shape[0] > 20000 else X_pca2_full_featA.shape[0]
idx = rng.choice(X_pca2_full_featA.shape[0], size=sample_n, replace=False)
X_samp = X_pca2_full_featA[idx]

for k in ks:
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    km.fit(X_pca2_full_featA)
    sil = silhouette_score(X_samp, km.predict(X_samp))
    sil_scores.append(sil)
    print(f"k={k:2d} | silhouette(sample)={sil:.4f}")

sil_full_featA_df = pd.DataFrame({"k": ks, "silhouette_sample": sil_scores})

plt.figure()
plt.plot(sil_full_featA_df["k"], sil_full_featA_df["silhouette_sample"], marker="o")
plt.xlabel("Number of clusters (k)")
plt.ylabel("Silhouette score (sample)")
plt.title("Silhouette (sample) — KMeans on PCA(2) (FULL engineered, TRAIN)")
plt.tight_layout()
plt.savefig(FIG_DIR / "kmeans_silhouette_full_featA.png", dpi=300)
plt.show()

In [ ]:
K_FULL_FEATA = 4

km_full_featA = KMeans(n_clusters=K_FULL_FEATA, random_state=42, n_init=10)
clusters_full_featA = km_full_featA.fit_predict(X_pca2_full_featA)

print("K_FULL_FEATA:", K_FULL_FEATA)
print("Cluster counts:", dict(zip(*np.unique(clusters_full_featA, return_counts=True))))

centroids = km_full_featA.cluster_centers_
cmap = plt.get_cmap("tab10")

plt.figure(figsize=(6,5))
for k in range(K_FULL_FEATA):
    m = clusters_full_featA == k
    plt.scatter(
        X_pca2_full_featA[m, 0],
        X_pca2_full_featA[m, 1],
        s=10, alpha=0.6,
        color=cmap(k),
        label=f"Cluster {k}"
    )

plt.scatter(
    centroids[:, 0], centroids[:, 1],
    marker="X", s=220,
    c=[cmap(i) for i in range(K_FULL_FEATA)],
    edgecolor="black", linewidths=1.5
)


plt.xlabel("PC1")
plt.ylabel("PC2")
plt.title("KMeans Clusters (k=4) — FULL engineered feature set (TRAIN)")
plt.legend(markerscale=2, frameon=True, ncol=2)
plt.tight_layout()
plt.savefig(FIG_DIR / "kmeans_clusters_full_featA_k4_centroids.png", dpi=300)
plt.show()

In [ ]:
# Attach clusters back to train data
df_train_full_featA_out = df_train_full_featA.copy()
df_train_full_featA_out["cluster_full_featA"] = clusters_full_featA

# Profile ALL engineered features (means)
profile_full_featA = (
    df_train_full_featA_out
    .groupby("cluster_full_featA")[FULL_COLS_FEATA]
    .mean()
    .round(3)
)

profile_full_featA

In [ ]:
# Diabetes prevalence (interpretation only)
df_train_full_featA_out[TARGET] = df_featA.loc[df_train_full_featA_out.index, TARGET]

diabetes_full_featA = (
    df_train_full_featA_out
    .groupby("cluster_full_featA")[TARGET]
    .mean()
    .round(3)
)

diabetes_full_featA

In [ ]:
# Diabetes prevalence (already computed)
base_prev = diabetes_full_base.rename("diabetes_prev_base")
featA_prev = diabetes_full_featA.rename("diabetes_prev_featA")

# Add a risk band label to make comparison human-readable
def risk_band(p):
    if p < 0.16: return "Low"
    if p < 0.22: return "Low–Mid"
    if p < 0.29: return "Mid–High"
    return "High"

base_tbl = base_prev.to_frame()
base_tbl["risk_band"] = base_tbl["diabetes_prev_base"].apply(risk_band)

featA_tbl = featA_prev.to_frame()
featA_tbl["risk_band"] = featA_tbl["diabetes_prev_featA"].apply(risk_band)

# Cluster sizes
base_sizes = pd.Series(df_train_full_base_out["cluster_full_base"].value_counts(), name="n_base")
featA_sizes = pd.Series(df_train_full_featA_out["cluster_full_featA"].value_counts(), name="n_featA")

# Combine
summary_base = base_tbl.join(base_sizes, how="left")
summary_featA = featA_tbl.join(featA_sizes, how="left")

display(summary_base.sort_values("diabetes_prev_base"))
display(summary_featA.sort_values("diabetes_prev_featA"))